# Face Detection and IoU Tracking — YOLO11 + IouTracker

| Stage | Model | Output |
|---|---|---|
| Face detection | `YoloFace11Detector` | per-frame bounding boxes + 5-pt keypoints |
| Tracking | `IouTracker` | persistent track IDs across frames |
| Merge | `tracker.merge()` | fuse short tracks from the same person |

`IouTracker` greedily assigns each detection to the open track with the highest IoU.
A new track is started when no suitable candidate is found.
Demo uses `multispeaker_short.mp4` (multi-speaker video fixture).

In [1]:
from exordium import FIXTURE_DIR
from exordium.video.core.io import get_video_metadata

video_path = FIXTURE_DIR / "video" / "multispeaker_short.mp4"
print(f"Video: {video_path.name}  exists={video_path.exists()}")

meta = get_video_metadata(video_path)
print(
    f"  {meta['num_frames']} frames @ {meta['fps']:.1f} fps  "
    f"({meta['width']}×{meta['height']})  {meta['duration']:.2f}s"
)

/Volumes/UGREEN/Dev/exordium/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Video: multispeaker_short.mp4  exists=True
  30 frames @ 30.0 fps  (1280×720)  1.00s


objc[7021]: Class AVFFrameReceiver is implemented in both /Volumes/UGREEN/Dev/exordium/.venv/lib/python3.14/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x10d2603a8) and /opt/homebrew/Cellar/ffmpeg/8.1.2/lib/libavdevice.62.3.102.dylib (0x12ddcc328). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[7021]: Class AVFAudioReceiver is implemented in both /Volumes/UGREEN/Dev/exordium/.venv/lib/python3.14/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x10d2603f8) and /opt/homebrew/Cellar/ffmpeg/8.1.2/lib/libavdevice.62.3.102.dylib (0x12ddcc378). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.


## Load detector

In [2]:
from exordium.video.face.detector.yolo11 import YoloFace11Detector

detector = YoloFace11Detector(device_id=None, conf=0.7)

2026-07-07 20:48:52 INFO YoloFace11Detector loaded 'yolo11n-pose_widerface' on cpu (conf=0.7).


## Step 1 — Detect faces in video

In [3]:
# detect_video streams frames in batches and returns VideoDetections
video_dets = detector.detect_video(video_path)

total_dets = sum(len(fd) for fd in video_dets)
frames_w_det = sum(1 for fd in video_dets if len(fd) > 0)

print(f"Total frames processed  : {len(video_dets)}")
print(f"Frames with ≥1 detection: {frames_w_det}")
print(f"Total face detections   : {total_dets}")

Total frames processed  : 30
Frames with ≥1 detection: 30
Total face detections   : 90


## Step 2 — Inspect a single detection

In [4]:
# Take the first frame that has at least one detection
fd = next(fd for fd in video_dets if len(fd) > 0)
det = list(fd)[0]

print(f"Frame id  : {det.frame_id}")
print(f"Score     : {det.score:.3f}")
print(f"BB (xywh) : {det.bb_xywh.tolist()}")
print(f"BB (xyxy) : {[int(v) for v in det.bb_xyxy.tolist()]}")
print(f"Landmarks : shape={det.landmarks.shape}  (right_eye, left_eye, nose, mouth_r, mouth_l)")

Frame id  : 0
Score     : 0.853
BB (xywh) : [936, 246, 105, 137]
BB (xyxy) : [936, 246, 1041, 383]
Landmarks : shape=torch.Size([5, 2])  (right_eye, left_eye, nose, mouth_r, mouth_l)


## Step 3 — IoU tracking

`IouTracker.label` iterates over all frames and greedily assigns each detection
to an existing open track or opens a new one.

In [5]:
from exordium.video.core import IouTracker

tracker = IouTracker(verbose=False, max_lost=10, iou_threshold=0.3)
tracker.label(video_dets, min_score=0.7, max_lost=10, iou_threshold=0.3)

tracks = tracker.tracks  # dict[int, Track]
print(f"Tracks created: {len(tracks)}")

Tracks created: 3


## Step 4 — Track analysis

In [6]:
lengths = {tid: len(t.detections) for tid, t in tracks.items()}
sorted_lengths = sorted(lengths.values())

robust = {tid: t for tid, t in tracks.items() if len(t.detections) >= 3}
short = {tid: t for tid, t in tracks.items() if len(t.detections) < 3}

print(
    f"Track length — min: {sorted_lengths[0]}  "
    f"median: {sorted_lengths[len(sorted_lengths) // 2]}  "
    f"max: {sorted_lengths[-1]}"
)
print(f"Robust tracks (≥ 3 frames): {len(robust)}")
print(f"Short  tracks (<  3 frames): {len(short)}  (likely false positives)")

Track length — min: 30  median: 30  max: 30
Robust tracks (≥ 3 frames): 3
Short  tracks (<  3 frames): 0  (likely false positives)


## Step 5 — Top tracks

In [7]:
top = sorted(tracks.items(), key=lambda kv: len(kv[1].detections), reverse=True)[:5]

print(f"{'Track':>6}  {'Frames':>6}  {'Start':>6}  {'End':>6}")
print("-" * 32)
for tid, t in top:
    print(
        f"{tid:>6d}  {len(t.detections):>6d}  "
        f"{t.first_detection().frame_id:>6d}  {t.last_detection().frame_id:>6d}"
    )

 Track  Frames   Start     End
--------------------------------
     0      30       0      29
     1      30       0      29
     2      30       0      29


## Step 6 — Merge short tracks

`tracker.merge()` applies the `merge_rule` to fuse nearby tracks that likely
belong to the same person (same IoU overlap, within max-lost gap).

In [8]:
n_before = len(tracker.tracks)
tracker.merge()
n_after = len(tracker.tracks)

print(f"Tracks before merge: {n_before}")
print(f"Tracks after  merge: {n_after}")

# Final summary
print()
print(f"Video: {meta['num_frames']} frames @ {meta['fps']:.1f} fps")
print(f"Detections: {total_dets} total  ({frames_w_det} frames with faces)")
print(f"Unique tracks (merged): {n_after}")

Tracks before merge: 3
Tracks after  merge: 3

Video: 30 frames @ 30.0 fps
Detections: 90 total  (30 frames with faces)
Unique tracks (merged): 3
